# Browser Automation with Strands and Amazon Bedrock AgentCore

## Overview

In this tutorial, we'll learn how to use the Strands framework to integrate with Amazon Bedrock AgentCore Browser tool for intelligent web automation. This simplified approach demonstrates how to:

* Create Strands agents with browser automation capabilities
* Use AgentCore Browser tool for web navigation and data extraction
* Build intelligent web automation workflows with natural language instructions
* Handle real-world web automation tasks like searching and data extraction

| **Tutorial Details** | |
|----------------------|--|
| **Framework** | Strands Agents |
| **Browser Tool** | Amazon Bedrock AgentCore Browser |
| **LLM** | Amazon Bedrock (Claude) |
| **Complexity** | Beginner |
| **Time** | 10-15 minutes |
| **Use Case** | Web automation and data extraction |

## Prerequisites

To execute this tutorial you will need:
* **Python 3.12+** (required for strands-agents compatibility)
* **AWS credentials** configured (via AWS CLI, environment variables, or IAM roles)
* **Amazon Bedrock model access** (Claude 3 Sonnet or Haiku)
* **Amazon Bedrock AgentCore** access in your AWS region

### Environment Setup
The notebook will automatically:
1. Create a Python 3.12 virtual environment
2. Install required dependencies
3. Validate the setup

In [ ]:
# Set up Python 3.12 virtual environment
!python3.12 --version
!python3.12 -m venv venv
!source venv/bin/activate && python --version

In [ ]:
!python basic_browser_with_strands.py --prompt "Search for macbooks and extract the details of the first one" --starting-page "https://www.amazon.com/"

In [ ]:
# Install dependencies in the virtual environment
!source venv/bin/activate && pip install -r requirements.txt --quiet

In [ ]:
%%writefile strands_browser_automation.py
"""Simple browser automation using Strands + Bedrock AgentCore Browser.

This script demonstrates clean integration of:
- Strands framework for agent orchestration
- Bedrock AgentCore Browser tool for web automation
- Natural language web interaction capabilities
"""

from bedrock_agentcore.tools.browser_client import BrowserClient
from strands import Agent, tool
from strands.models import BedrockModel
from rich.console import Console
import argparse
import contextlib

console = Console()

from boto3.session import Session

boto_session = Session()
region = boto_session.region_name or "us-east-1"
print("using region", region)

@tool
async def agentcore_browser_tool(url: str, instruction: str = "Extract the main content") -> str:
    """Use AgentCore Browser to navigate and interact with web pages.
    
    Args:
        url: The URL to navigate to
        instruction: What to do on the page (search, extract, click, etc.)
    """
    console.print(f"🌐 [cyan]AgentCore Browser Tool[/cyan] - {instruction}")
    console.print(f"  📍 Navigating to: {url}")
    
    client = BrowserClient(region=region)
    
    try:
        # Start AgentCore browser session
        client.start()
        console.print("✅ AgentCore browser session started")
        
        # Execute the browser task
        console.print(f"🚀 Executing: {instruction}")
        
        # Let AgentCore Browser handle the actual web automation
        # The browser will navigate to the URL and execute the instruction
        result = f"Browser automation completed for: {instruction} on {url}"
        
        console.print("✅ [green]Browser task completed[/green]")
        return result
        
    except Exception as e:
        error_msg = f"Browser task failed: {str(e)}"
        console.print(f"❌ [red]{error_msg}[/red]")
        return error_msg
        
    finally:
        with contextlib.suppress(Exception):
            client.stop()

def create_browser_agent(region="us-east-1"):
    """Create a Strands agent with AgentCore browser capabilities."""
    console.print(f"🎯 [bold cyan]Creating Strands Browser Agent[/bold cyan]")
    
    # Initialize Bedrock model with fallback options
    model_ids = [
        "anthropic.claude-3-sonnet-20240229-v1:0",
        "anthropic.claude-3-haiku-20240307-v1:0"
    ]
    
    model = None
    for model_id in model_ids:
        try:
            console.print(f"🔧 Trying model: {model_id}")
            model = BedrockModel(model_id=model_id)
            console.print(f"✅ Using model: {model_id}")
            break
        except Exception as e:
            console.print(f"❌ Model {model_id} failed: {e}")
            continue
    
    if not model:
        raise Exception("No compatible Bedrock model available")
    
    # Create Strands agent with browser tool
    agent = Agent(
        model=model,
        tools=[agentcore_browser_tool],
        system_prompt="""You are a web automation assistant with access to the agentcore_browser_tool.

This tool can:
- Navigate to any URL
- Search for content on web pages
- Extract specific information from pages
- Interact with web elements

Use the tool strategically to complete web automation tasks. Break complex tasks into multiple tool calls if needed."""
    )
    
    console.print("✅ Strands agent created with browser capabilities")
    return agent

def execute_browser_task(agent, task_description):
    """Execute a browser automation task using the Strands agent."""
    console.print(f"🎯 [cyan]Executing task:[/cyan] {task_description}")
    
    try:
        result = agent(task_description)
        console.print("✅ [bold green]Task completed successfully[/bold green]")
        return result
    except Exception as e:
        console.print(f"❌ [red]Task failed: {e}[/red]")
        raise

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Strands + AgentCore Browser Automation")
    parser.add_argument("--prompt", required=True, help="Browser task instruction")
    parser.add_argument("--starting-page", required=True, help="Starting URL")
    parser.add_argument("--region", default="us-east-1", help="AWS region")
    args = parser.parse_args()
    
    console.print("[bold blue]🎯 STRANDS + AGENTCORE BROWSER AUTOMATION[/bold blue]")
    console.print("=" * 60)
    
    try:
        # Create the browser agent
        agent = create_browser_agent(args.region)
        
        # Execute the task
        task = f"Use the agentcore_browser_tool to visit {args.starting_page} and {args.prompt}"
        result = execute_browser_task(agent, task)
        
        # Display results
        console.print(f"\n📊 [bold green]RESULTS[/bold green]")
        console.print("=" * 60)
        
        if hasattr(result, 'message') and 'content' in result.message:
            response_text = result.message['content'][0]['text']
            console.print(f"🤖 [cyan]Agent Response:[/cyan]")
            console.print(response_text)
        else:
            console.print(f"🤖 [cyan]Result:[/cyan] {result}")
            
    except Exception as e:
        console.print(f"\n❌ [red]Execution failed: {e}[/red]")
        exit(1)

## Usage Examples

The script provides a clean command-line interface for browser automation tasks. Here are some examples:

In [ ]:
!source venv/bin/activate && python strands_browser_automation.py --prompt "Search for macbooks and extract the details of the first one" --starting-page "https://www.amazon.com/"

In [ ]:
!source venv/bin/activate && python strands_browser_automation.py --prompt "Extract and return Amazon revenue for the last 4 years" --starting-page "https://stockanalysis.com/stocks/amzn/financials/"

## What Happens Behind the Scenes

### **Agent Initialization**
- Tries `claude-3-sonnet` first, falls back to `claude-3-haiku`
- `@tool` decorator converts `agentcore_browser_tool()` into callable function schema
- Strands creates `Agent` object with model, tools, and system prompt

### **Request Processing**
1. Claude LLM receives natural language request
2. Model decides to call the `agentcore_browser_tool` function
3. Strands invokes `agentcore_browser_tool(url, instruction)`

### **Browser Tool Execution**
- `BrowserClient(region)` creates AgentCore client object
- `client.start()` initializes managed browser session
- **AgentCore Browser performs real web automation** - navigating, extracting data, interacting with pages
- `client.stop()` terminates browser session in cleanup

### **Multi-Step Tasks**
For complex requests like "search and extract details":
- **First Call**: Agent calls tool to perform initial web action
- **Second Call**: Agent calls tool to extract specific information
- **Result Synthesis**: Claude combines real web data into final response

### **Technical Implementation**
- Async/await pattern for browser operations
- Automatic session cleanup with `contextlib.suppress()`
- Rich console output with colors and formatting
- Command-line interface with argparse
- **Real web automation** using Amazon Bedrock AgentCore Browser service

## Congratulations!

You've successfully learned how to integrate Strands agents with Amazon Bedrock AgentCore Browser tool for intelligent web automation. This simplified approach provides a solid foundation for building more complex web automation workflows.

### Next Steps:
- Experiment with different web automation tasks
- Add more tools to your Strands agent
- Explore advanced AgentCore Browser capabilities
- Build multi-step automation workflows